In [1]:
code = '''import torch, torch.nn as nn
import snntorch as snn
from snntorch import surrogate

class SNN(nn.Module):
    def __init__(self, n_in=128, n_h=128, n_out=2, beta=0.9):
        super().__init__()
        g = surrogate.fast_sigmoid()
        self.fc1 = nn.Linear(n_in, n_h); self.lif1 = snn.Leaky(beta=beta, spike_grad=g)
        self.fc2 = nn.Linear(n_h, n_out); self.lif2 = snn.Leaky(beta=beta, spike_grad=g)
        self.drop = nn.Dropout(0.3)

    def forward(self, x):
        mem1 = self.lif1.init_leaky(); mem2 = self.lif2.init_leaky()
        out, hid = [], []
        for t in range(x.shape[0]):
            spk1, mem1 = self.lif1(self.fc1(x[t]), mem1)
            spk2, mem2 = self.lif2(self.fc2(self.drop(spk1)), mem2)
            out.append(spk2); hid.append(spk1)
        return torch.stack(out), torch.stack(hid)
'''
with open("snn_model.py", "w") as f:
    f.write(code)
print("saved")

saved


In [2]:
from snn_model import SNN
import snntorch.functional as SF
from snntorch import spikegen

In [3]:
import numpy as np, torch
from preprocess import band_power_features
from sklearn.model_selection import GroupShuffleSplit

z = np.load("data/processed/s01.npz")
F = band_power_features(z["X"]).astype(np.float32)
y = z["y"][:, 0]                      # 0 = valence, 1 = arousal
tid = z["trial_id"]

tr, te = next(GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=0).split(F, y, tid))

mu, sd = F[tr].mean(0), F[tr].std(0) + 1e-8        # statistics from TRAIN only
def scale(a):                                       # map to [0, 1] for rate coding
    return np.clip((a - mu) / sd, -3, 3) / 6 + 0.5

Xtr = torch.tensor(scale(F[tr]), dtype=torch.float32)
Xte = torch.tensor(scale(F[te]), dtype=torch.float32)
ytr = torch.tensor(y[tr]); yte = torch.tensor(y[te])
print(Xtr.shape, Xte.shape)

torch.Size([870, 128]) torch.Size([290, 128])


In [4]:
T = 25                                              # time steps per sample
model = SNN()
opt = torch.optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-4)
loss_fn = SF.ce_rate_loss()

def evaluate(X, Y):
    model.eval()
    with torch.no_grad():
        spk, hid = model(spikegen.rate(X, num_steps=T))
    return SF.accuracy_rate(spk, Y), hid.mean().item()

for epoch in range(30):
    model.train()
    perm = torch.randperm(len(Xtr))
    for i in range(0, len(perm), 64):
        idx = perm[i:i+64]
        spk, _ = model(spikegen.rate(Xtr[idx], num_steps=T))
        loss = loss_fn(spk, ytr[idx])
        opt.zero_grad(); loss.backward(); opt.step()
    if epoch % 5 == 4:
        tr_acc, _ = evaluate(Xtr, ytr); te_acc, rate = evaluate(Xte, yte)
        print(f"epoch {epoch+1}  train {tr_acc:.3f}  test {te_acc:.3f}  hidden firing rate {rate:.3f}")

epoch 5  train 0.567  test 0.400  hidden firing rate 0.127
epoch 10  train 0.567  test 0.400  hidden firing rate 0.144
epoch 15  train 0.567  test 0.400  hidden firing rate 0.153
epoch 20  train 0.567  test 0.400  hidden firing rate 0.173
epoch 25  train 0.567  test 0.400  hidden firing rate 0.180
epoch 30  train 0.621  test 0.348  hidden firing rate 0.178


In [1]:
import numpy as np, torch, time
print("torch imported", flush=True)

from snn_model import SNN2
print("snn_model imported", flush=True)

z = np.load("data/processed/features_all.npz")
F, Y, S = z["F"], z["Y"], z["S"]
print("data loaded", F.shape, flush=True)

model = SNN2(T=15)
x = torch.tensor(F[:256])
t0 = time.time()
spk, hid = model(x)
print("one batch forward:", round(time.time() - t0, 2), "seconds", spk.shape, flush=True)

torch imported
snn_model imported
data loaded (37120, 128)
one batch forward: 0.06 seconds torch.Size([15, 256, 2])


In [2]:
import numpy as np, torch, time
import snntorch.functional as SF
from snn_model import SNN2
from sklearn.metrics import balanced_accuracy_score, f1_score

z = np.load("data/processed/features_all.npz")
F, Y, S = z["F"], z["Y"], z["S"]
col = 0                                        # 0 = valence, 1 = arousal

tr, te = S <= 24, S > 24
Xtr = torch.tensor(F[tr]); ytr = torch.tensor(Y[tr, col])
Xte = torch.tensor(F[te]); yte = torch.tensor(Y[te, col])

torch.manual_seed(0)
model = SNN2(T=15)                             # fewer time steps = faster
opt = torch.optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-3)
loss_fn = SF.ce_rate_loss()

def predict(X, bs=1024):
    model.eval()
    preds, rates = [], []
    with torch.no_grad():
        for i in range(0, len(X), bs):
            spk, hid = model(X[i:i+bs])
            preds.append(spk.sum(0).argmax(1)); rates.append(hid.mean().item())
    return torch.cat(preds), float(np.mean(rates))

sub = torch.randperm(len(Xtr))[:3000]          # small subset for the train score

for epoch in range(10):
    t0 = time.time()
    model.train()
    perm = torch.randperm(len(Xtr))
    for i in range(0, len(perm), 256):
        idx = perm[i:i+256]
        spk, _ = model(Xtr[idx])
        loss = loss_fn(spk, ytr[idx])
        opt.zero_grad(); loss.backward(); opt.step()
    ptr, _ = predict(Xtr[sub]); pte, rate = predict(Xte)
    tr_acc = (ptr == ytr[sub]).float().mean().item()
    te_acc = (pte == yte).float().mean().item()
    bal = balanced_accuracy_score(yte.numpy(), pte.numpy())
    print(f"epoch {epoch+1:2d}  train {tr_acc:.3f}  test {te_acc:.3f}  "
          f"balanced {bal:.3f}  rate {rate:.3f}  ({time.time()-t0:.0f}s)", flush=True)

epoch  1  train 0.593  test 0.521  balanced 0.520  rate 0.047  (7s)
epoch  2  train 0.603  test 0.499  balanced 0.507  rate 0.051  (7s)
epoch  3  train 0.616  test 0.518  balanced 0.515  rate 0.049  (6s)
epoch  4  train 0.625  test 0.506  balanced 0.510  rate 0.048  (7s)
epoch  5  train 0.630  test 0.498  balanced 0.513  rate 0.050  (7s)
epoch  6  train 0.643  test 0.520  balanced 0.515  rate 0.047  (6s)
epoch  7  train 0.645  test 0.548  balanced 0.519  rate 0.043  (6s)
epoch  8  train 0.638  test 0.516  balanced 0.516  rate 0.042  (6s)
epoch  9  train 0.636  test 0.559  balanced 0.514  rate 0.040  (7s)
epoch 10  train 0.656  test 0.535  balanced 0.523  rate 0.033  (6s)


In [3]:
import numpy as np, torch, time
import snntorch.functional as SF
from snn_model import SNN2
from sklearn.metrics import balanced_accuracy_score, f1_score

z = np.load("data/processed/features_all.npz")
F, Y, S = z["F"], z["Y"], z["S"]
col = 1                                        # 0 = valence, 1 = arousal

tr, te = S <= 24, S > 24
Xtr = torch.tensor(F[tr]); ytr = torch.tensor(Y[tr, col])
Xte = torch.tensor(F[te]); yte = torch.tensor(Y[te, col])

torch.manual_seed(0)
model = SNN2(T=15)                             # fewer time steps = faster
opt = torch.optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-3)
loss_fn = SF.ce_rate_loss()

def predict(X, bs=1024):
    model.eval()
    preds, rates = [], []
    with torch.no_grad():
        for i in range(0, len(X), bs):
            spk, hid = model(X[i:i+bs])
            preds.append(spk.sum(0).argmax(1)); rates.append(hid.mean().item())
    return torch.cat(preds), float(np.mean(rates))

sub = torch.randperm(len(Xtr))[:3000]          # small subset for the train score

for epoch in range(10):
    t0 = time.time()
    model.train()
    perm = torch.randperm(len(Xtr))
    for i in range(0, len(perm), 256):
        idx = perm[i:i+256]
        spk, _ = model(Xtr[idx])
        loss = loss_fn(spk, ytr[idx])
        opt.zero_grad(); loss.backward(); opt.step()
    ptr, _ = predict(Xtr[sub]); pte, rate = predict(Xte)
    tr_acc = (ptr == ytr[sub]).float().mean().item()
    te_acc = (pte == yte).float().mean().item()
    bal = balanced_accuracy_score(yte.numpy(), pte.numpy())
    print(f"epoch {epoch+1:2d}  train {tr_acc:.3f}  test {te_acc:.3f}  "
          f"balanced {bal:.3f}  rate {rate:.3f}  ({time.time()-t0:.0f}s)", flush=True)

epoch  1  train 0.536  test 0.499  balanced 0.518  rate 0.055  (6s)
epoch  2  train 0.517  test 0.486  balanced 0.517  rate 0.050  (8s)
epoch  3  train 0.583  test 0.536  balanced 0.523  rate 0.037  (9s)
epoch  4  train 0.589  test 0.549  balanced 0.540  rate 0.035  (7s)
epoch  5  train 0.581  test 0.530  balanced 0.538  rate 0.038  (7s)
epoch  6  train 0.612  test 0.549  balanced 0.531  rate 0.032  (6s)
epoch  7  train 0.602  test 0.558  balanced 0.540  rate 0.031  (7s)
epoch  8  train 0.598  test 0.548  balanced 0.524  rate 0.029  (6s)
epoch  9  train 0.615  test 0.549  balanced 0.525  rate 0.029  (5s)
epoch 10  train 0.628  test 0.546  balanced 0.517  rate 0.027  (5s)


In [4]:
import numpy as np, torch
import snntorch.functional as SF
from snn_model import SNN2
from sklearn.model_selection import GroupKFold
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import balanced_accuracy_score

z = np.load("data/processed/features_all.npz")
F, Y, S = z["F"], z["Y"], z["S"]
col = 0                      # 0 = valence, 1 = arousal
subjects = range(1, 9)       # first 8; later use range(1, 33)

def train_snn(Xtr, ytr, epochs=20):
    torch.manual_seed(0)
    m = SNN2(n_h=128, T=15)
    opt = torch.optim.Adam(m.parameters(), lr=1e-3, weight_decay=1e-3)
    lf = SF.ce_rate_loss()
    for _ in range(epochs):
        m.train()
        perm = torch.randperm(len(Xtr))
        for i in range(0, len(perm), 128):
            idx = perm[i:i+128]
            spk, _ = m(Xtr[idx])
            loss = lf(spk, ytr[idx])
            opt.zero_grad(); loss.backward(); opt.step()
    return m

snn_bal, lr_bal, rates = [], [], []
for sid in subjects:
    Fs, ys = F[S == sid], Y[S == sid, col]
    trial = np.arange(len(Fs)) // 29          # 29 windows per trial
    sb, lb, rt = [], [], []
    for tr_i, te_i in GroupKFold(n_splits=4).split(Fs, ys, trial):
        if len(np.unique(ys[tr_i])) < 2 or len(np.unique(ys[te_i])) < 2:
            continue                          # skip folds with one class
        Xtr, ytr = torch.tensor(Fs[tr_i]), torch.tensor(ys[tr_i])
        Xte = torch.tensor(Fs[te_i])
        m = train_snn(Xtr, ytr)
        m.eval()
        with torch.no_grad():
            spk, hid = m(Xte)
        pred = spk.sum(0).argmax(1).numpy()
        sb.append(balanced_accuracy_score(ys[te_i], pred))
        rt.append(hid.mean().item())
        lr = LogisticRegression(max_iter=1000).fit(Fs[tr_i], ys[tr_i])
        lb.append(balanced_accuracy_score(ys[te_i], lr.predict(Fs[te_i])))
    if sb:
        snn_bal.append(np.mean(sb)); lr_bal.append(np.mean(lb)); rates.append(np.mean(rt))
        print(f"subject {sid:2d}  SNN {snn_bal[-1]:.3f}  LogReg {lr_bal[-1]:.3f}  rate {rates[-1]:.3f}", flush=True)

print("MEAN  SNN balanced:", round(np.mean(snn_bal), 3),
      "| LogReg balanced:", round(np.mean(lr_bal), 3),
      "| firing rate:", round(np.mean(rates), 3))

subject  1  SNN 0.491  LogReg 0.503  rate 0.145
subject  2  SNN 0.654  LogReg 0.618  rate 0.175
subject  3  SNN 0.708  LogReg 0.689  rate 0.161
subject  4  SNN 0.639  LogReg 0.590  rate 0.170
subject  5  SNN 0.581  LogReg 0.585  rate 0.147
subject  6  SNN 0.480  LogReg 0.472  rate 0.155
subject  7  SNN 0.451  LogReg 0.456  rate 0.167
subject  8  SNN 0.553  LogReg 0.566  rate 0.167
MEAN  SNN balanced: 0.57 | LogReg balanced: 0.56 | firing rate: 0.161


In [5]:
import numpy as np, torch
import snntorch.functional as SF
from snn_model import SNN2
from sklearn.model_selection import GroupKFold
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import balanced_accuracy_score

z = np.load("data/processed/features_all.npz")
F, Y, S = z["F"], z["Y"], z["S"]
col = 1                      # 0 = valence, 1 = arousal
subjects = range(1, 9)       # first 8; later use range(1, 33)

def train_snn(Xtr, ytr, epochs=20):
    torch.manual_seed(0)
    m = SNN2(n_h=128, T=15)
    opt = torch.optim.Adam(m.parameters(), lr=1e-3, weight_decay=1e-3)
    lf = SF.ce_rate_loss()
    for _ in range(epochs):
        m.train()
        perm = torch.randperm(len(Xtr))
        for i in range(0, len(perm), 128):
            idx = perm[i:i+128]
            spk, _ = m(Xtr[idx])
            loss = lf(spk, ytr[idx])
            opt.zero_grad(); loss.backward(); opt.step()
    return m

snn_bal, lr_bal, rates = [], [], []
for sid in subjects:
    Fs, ys = F[S == sid], Y[S == sid, col]
    trial = np.arange(len(Fs)) // 29          # 29 windows per trial
    sb, lb, rt = [], [], []
    for tr_i, te_i in GroupKFold(n_splits=4).split(Fs, ys, trial):
        if len(np.unique(ys[tr_i])) < 2 or len(np.unique(ys[te_i])) < 2:
            continue                          # skip folds with one class
        Xtr, ytr = torch.tensor(Fs[tr_i]), torch.tensor(ys[tr_i])
        Xte = torch.tensor(Fs[te_i])
        m = train_snn(Xtr, ytr)
        m.eval()
        with torch.no_grad():
            spk, hid = m(Xte)
        pred = spk.sum(0).argmax(1).numpy()
        sb.append(balanced_accuracy_score(ys[te_i], pred))
        rt.append(hid.mean().item())
        lr = LogisticRegression(max_iter=1000).fit(Fs[tr_i], ys[tr_i])
        lb.append(balanced_accuracy_score(ys[te_i], lr.predict(Fs[te_i])))
    if sb:
        snn_bal.append(np.mean(sb)); lr_bal.append(np.mean(lb)); rates.append(np.mean(rt))
        print(f"subject {sid:2d}  SNN {snn_bal[-1]:.3f}  LogReg {lr_bal[-1]:.3f}  rate {rates[-1]:.3f}", flush=True)

print("MEAN  SNN balanced:", round(np.mean(snn_bal), 3),
      "| LogReg balanced:", round(np.mean(lr_bal), 3),
      "| firing rate:", round(np.mean(rates), 3))

subject  1  SNN 0.593  LogReg 0.598  rate 0.141
subject  2  SNN 0.533  LogReg 0.478  rate 0.164
subject  3  SNN 0.479  LogReg 0.553  rate 0.143
subject  4  SNN 0.605  LogReg 0.603  rate 0.166
subject  5  SNN 0.507  LogReg 0.486  rate 0.147
subject  6  SNN 0.539  LogReg 0.496  rate 0.160
subject  7  SNN 0.499  LogReg 0.545  rate 0.168
subject  8  SNN 0.565  LogReg 0.554  rate 0.154
MEAN  SNN balanced: 0.54 | LogReg balanced: 0.539 | firing rate: 0.155


In [6]:
import numpy as np, torch
import snntorch.functional as SF
from snn_model import SNN2
from sklearn.model_selection import GroupKFold
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import balanced_accuracy_score

z = np.load("data/processed/features_all.npz")
F, Y, S = z["F"], z["Y"], z["S"]
col = 0                      # 0 = valence, 1 = arousal
subjects = range(1, 33)       # first 8; later use range(1, 33)

def train_snn(Xtr, ytr, epochs=20):
    torch.manual_seed(0)
    m = SNN2(n_h=128, T=15)
    opt = torch.optim.Adam(m.parameters(), lr=1e-3, weight_decay=1e-3)
    lf = SF.ce_rate_loss()
    for _ in range(epochs):
        m.train()
        perm = torch.randperm(len(Xtr))
        for i in range(0, len(perm), 128):
            idx = perm[i:i+128]
            spk, _ = m(Xtr[idx])
            loss = lf(spk, ytr[idx])
            opt.zero_grad(); loss.backward(); opt.step()
    return m

snn_bal, lr_bal, rates = [], [], []
for sid in subjects:
    Fs, ys = F[S == sid], Y[S == sid, col]
    trial = np.arange(len(Fs)) // 29          # 29 windows per trial
    sb, lb, rt = [], [], []
    for tr_i, te_i in GroupKFold(n_splits=4).split(Fs, ys, trial):
        if len(np.unique(ys[tr_i])) < 2 or len(np.unique(ys[te_i])) < 2:
            continue                          # skip folds with one class
        Xtr, ytr = torch.tensor(Fs[tr_i]), torch.tensor(ys[tr_i])
        Xte = torch.tensor(Fs[te_i])
        m = train_snn(Xtr, ytr)
        m.eval()
        with torch.no_grad():
            spk, hid = m(Xte)
        pred = spk.sum(0).argmax(1).numpy()
        sb.append(balanced_accuracy_score(ys[te_i], pred))
        rt.append(hid.mean().item())
        lr = LogisticRegression(max_iter=1000).fit(Fs[tr_i], ys[tr_i])
        lb.append(balanced_accuracy_score(ys[te_i], lr.predict(Fs[te_i])))
    if sb:
        snn_bal.append(np.mean(sb)); lr_bal.append(np.mean(lb)); rates.append(np.mean(rt))
        print(f"subject {sid:2d}  SNN {snn_bal[-1]:.3f}  LogReg {lr_bal[-1]:.3f}  rate {rates[-1]:.3f}", flush=True)

print("MEAN  SNN balanced:", round(np.mean(snn_bal), 3),
      "| LogReg balanced:", round(np.mean(lr_bal), 3),
      "| firing rate:", round(np.mean(rates), 3))

subject  1  SNN 0.491  LogReg 0.503  rate 0.145
subject  2  SNN 0.654  LogReg 0.618  rate 0.175
subject  3  SNN 0.708  LogReg 0.689  rate 0.161
subject  4  SNN 0.639  LogReg 0.590  rate 0.170
subject  5  SNN 0.581  LogReg 0.585  rate 0.147
subject  6  SNN 0.480  LogReg 0.472  rate 0.155
subject  7  SNN 0.451  LogReg 0.456  rate 0.167
subject  8  SNN 0.553  LogReg 0.566  rate 0.167
subject  9  SNN 0.660  LogReg 0.629  rate 0.164
subject 10  SNN 0.759  LogReg 0.767  rate 0.167
subject 11  SNN 0.527  LogReg 0.539  rate 0.162
subject 12  SNN 0.606  LogReg 0.618  rate 0.156
subject 13  SNN 0.760  LogReg 0.684  rate 0.159
subject 14  SNN 0.744  LogReg 0.723  rate 0.165
subject 15  SNN 0.717  LogReg 0.683  rate 0.167
subject 16  SNN 0.658  LogReg 0.552  rate 0.142
subject 17  SNN 0.562  LogReg 0.551  rate 0.151
subject 18  SNN 0.710  LogReg 0.689  rate 0.172
subject 19  SNN 0.555  LogReg 0.545  rate 0.144
subject 20  SNN 0.675  LogReg 0.641  rate 0.166
subject 21  SNN 0.580  LogReg 0.544  rat

In [7]:
import numpy as np, torch
import snntorch.functional as SF
from snn_model import SNN2
from sklearn.model_selection import GroupKFold
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import balanced_accuracy_score

z = np.load("data/processed/features_all.npz")
F, Y, S = z["F"], z["Y"], z["S"]
col = 1                      # 0 = valence, 1 = arousal
subjects = range(1, 33)       

def train_snn(Xtr, ytr, epochs=20):
    torch.manual_seed(0)
    m = SNN2(n_h=128, T=15)
    opt = torch.optim.Adam(m.parameters(), lr=1e-3, weight_decay=1e-3)
    lf = SF.ce_rate_loss()
    for _ in range(epochs):
        m.train()
        perm = torch.randperm(len(Xtr))
        for i in range(0, len(perm), 128):
            idx = perm[i:i+128]
            spk, _ = m(Xtr[idx])
            loss = lf(spk, ytr[idx])
            opt.zero_grad(); loss.backward(); opt.step()
    return m

snn_bal, lr_bal, rates = [], [], []
for sid in subjects:
    Fs, ys = F[S == sid], Y[S == sid, col]
    trial = np.arange(len(Fs)) // 29          # 29 windows per trial
    sb, lb, rt = [], [], []
    for tr_i, te_i in GroupKFold(n_splits=4).split(Fs, ys, trial):
        if len(np.unique(ys[tr_i])) < 2 or len(np.unique(ys[te_i])) < 2:
            continue                          # skip folds with one class
        Xtr, ytr = torch.tensor(Fs[tr_i]), torch.tensor(ys[tr_i])
        Xte = torch.tensor(Fs[te_i])
        m = train_snn(Xtr, ytr)
        m.eval()
        with torch.no_grad():
            spk, hid = m(Xte)
        pred = spk.sum(0).argmax(1).numpy()
        sb.append(balanced_accuracy_score(ys[te_i], pred))
        rt.append(hid.mean().item())
        lr = LogisticRegression(max_iter=1000).fit(Fs[tr_i], ys[tr_i])
        lb.append(balanced_accuracy_score(ys[te_i], lr.predict(Fs[te_i])))
    if sb:
        snn_bal.append(np.mean(sb)); lr_bal.append(np.mean(lb)); rates.append(np.mean(rt))
        print(f"subject {sid:2d}  SNN {snn_bal[-1]:.3f}  LogReg {lr_bal[-1]:.3f}  rate {rates[-1]:.3f}", flush=True)

print("MEAN  SNN balanced:", round(np.mean(snn_bal), 3),
      "| LogReg balanced:", round(np.mean(lr_bal), 3),
      "| firing rate:", round(np.mean(rates), 3))

subject  1  SNN 0.593  LogReg 0.598  rate 0.141
subject  2  SNN 0.533  LogReg 0.478  rate 0.164
subject  3  SNN 0.479  LogReg 0.553  rate 0.143
subject  4  SNN 0.605  LogReg 0.603  rate 0.166
subject  5  SNN 0.507  LogReg 0.486  rate 0.147
subject  6  SNN 0.539  LogReg 0.496  rate 0.160
subject  7  SNN 0.499  LogReg 0.545  rate 0.168
subject  8  SNN 0.565  LogReg 0.554  rate 0.154
subject  9  SNN 0.510  LogReg 0.534  rate 0.174
subject 10  SNN 0.636  LogReg 0.656  rate 0.159
subject 11  SNN 0.475  LogReg 0.436  rate 0.170
subject 12  SNN 0.497  LogReg 0.500  rate 0.137
subject 13  SNN 0.487  LogReg 0.563  rate 0.158
subject 14  SNN 0.538  LogReg 0.541  rate 0.158
subject 15  SNN 0.642  LogReg 0.613  rate 0.173
subject 16  SNN 0.616  LogReg 0.616  rate 0.153
subject 17  SNN 0.678  LogReg 0.654  rate 0.156
subject 18  SNN 0.477  LogReg 0.527  rate 0.169
subject 19  SNN 0.469  LogReg 0.485  rate 0.141
subject 20  SNN 0.509  LogReg 0.539  rate 0.154
subject 21  SNN 0.502  LogReg 0.515  rat

In [8]:
import numpy as np, torch, os
import snntorch.functional as SF
from snn_model import SNN2

z = np.load("data/processed/features_all.npz")
F, Y, S = z["F"], z["Y"], z["S"]
tr, te = S <= 24, S > 24
os.makedirs("models", exist_ok=True)

def train(col, name, epochs=10):
    torch.manual_seed(0)
    m = SNN2(T=15)
    opt = torch.optim.Adam(m.parameters(), lr=1e-3, weight_decay=1e-3)
    lf = SF.ce_rate_loss()
    Xtr, ytr = torch.tensor(F[tr]), torch.tensor(Y[tr, col])
    for _ in range(epochs):
        m.train()
        perm = torch.randperm(len(Xtr))
        for i in range(0, len(perm), 256):
            idx = perm[i:i+256]
            spk, _ = m(Xtr[idx])
            loss = lf(spk, ytr[idx])
            opt.zero_grad(); loss.backward(); opt.step()
    torch.save(m.state_dict(), f"models/snn_{name}.pt")
    return m

def predict(m, X, bs=1024):
    m.eval(); out = []
    with torch.no_grad():
        for i in range(0, len(X), bs):
            spk, _ = m(X[i:i+bs])
            out.append(spk.sum(0).argmax(1))
    return torch.cat(out).numpy()

mv, ma = train(0, "valence"), train(1, "arousal")
Xte = torch.tensor(F[te])
pv, pa = predict(mv, Xte), predict(ma, Xte)

true_q = Y[te, 0] * 2 + Y[te, 1]
pred_q = pv * 2 + pa
print("4-emotion accuracy:", round((true_q == pred_q).mean(), 3), "(guessing = about 0.25-0.35)")

names = {3: "Happy / Excited", 2: "Calm / Relaxed", 1: "Angry / Stressed", 0: "Sad / Bored"}
print("Example predictions:", [names[q] for q in pred_q[:5]])

4-emotion accuracy: 0.29 (guessing = about 0.25-0.35)
Example predictions: ['Sad / Bored', 'Happy / Excited', 'Calm / Relaxed', 'Sad / Bored', 'Sad / Bored']
